# FATHOM - combine and scale

This notebook performs two steps:
1. Combine the three types of FATHOM flood (pluvial, fluvial, coastal) into a single layer of maximum flood depth.  
2. Scale the combined layer to match the 1km resolution GHS-POP layer, aggregating the data at the following classification  
  i. Proportion of 30m cells within each 1km grid cell where maximum flood depth exceeds 0.5m  
  ii. Proportion where maximum flood depth exceeds 0.15m  
  iii. Proportion where maximum flood depth exceeds 0m  

In [2]:
import os, time, io, json, sys
import urllib3
import boto3
import rasterio

import geopandas as gpd
import pandas as pd
import numpy as np

from functools import reduce
from urllib3.exceptions import InsecureRequestWarning
from botocore import UNSIGNED
from botocore.config import Config
from tqdm.notebook import tqdm
from shapely.geometry import box, mapping

urllib3.disable_warnings(InsecureRequestWarning)

def tPrint(s):
    """prints the time along with the message"""
    print("%s\t%s" % (time.strftime("%H:%M:%S"), s))

s3_client = boto3.client('s3', verify=False)

sys.path.insert(0, "C:/WBG/Work/Code/GOSTrocks/src")

import GOSTrocks.dataMisc as dataMisc
import GOSTrocks.rasterMisc as rMisc

#Mute runtime warnings for cleaner output
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

%load_ext autoreload
%autoreload 2    

In [ ]:
local_folder = "C:/WBG/Work/Projects/FATHOM_COLLAPSE"
out_folder = os.path.join(local_folder, "FATHOM_summaries")
map_folder = os.path.join(local_folder, "FATHOM_maps")
for tF in [out_folder, map_folder]:
    if not os.path.exists(tF):
        os.makedirs(tF)

ghs_pop_files = [
    "C:\\WBG\\Work\\data\\URBAN\\SMOD_POP\\GHS_POP_E2025_GLOBE_R2023A_54009_1000_V1_0.tif", 
    "C:\\WBG\\Work\\data\\URBAN\\SMOD_POP\\GHS_POP_E2030_GLOBE_R2023A_54009_100_V1_0.tif"
]

s3_bucket = "wbg-geography01"
s3_prefix = "FATHOM"
return_period = 100

In [4]:
# Use the S3 client to get a list of VRT files in the specified bucket and prefix
response = s3_client.list_objects_v2(Bucket=s3_bucket, Prefix=s3_prefix)
vrt_files = [obj['Key'] for obj in response.get('Contents', []) if obj['Key'].endswith('.vrt')]

# Turn the list of vrt files into a dataframe
vrt_breakdown = [[x] + x.split('-') for x in vrt_files]
vrt_df = pd.DataFrame(vrt_breakdown, columns=["path", 'prefix', 'res', 'offset', 'return_period', 'hazard', 'defended', 'metric', 'year', 'scenario', 'version', "other"])
vrt_df = vrt_df.loc[:, ['return_period', 'hazard', 'defended', 'year', 'scenario', "path"]]

# Focus on just the 100-year return period for now
vrt_df = vrt_df[vrt_df['return_period'] == f"1in{return_period}"]
# Drop the undefended models
vrt_df = vrt_df[vrt_df['defended'] == "DEFENDED"]
vrt_df

,return_period,hazard,defended,year,scenario,path
65,1in100,COASTAL,DEFENDED,2020,PERCENTILE50,FATHOM/FLOOD_MAP-1ARCSEC-NW_OFFSET-1in100-COAS...
66,1in100,COASTAL,DEFENDED,2030,SSP1_2.6,FATHOM/FLOOD_MAP-1ARCSEC-NW_OFFSET-1in100-COAS...
67,1in100,COASTAL,DEFENDED,2030,SSP2_4.5,FATHOM/FLOOD_MAP-1ARCSEC-NW_OFFSET-1in100-COAS...
68,1in100,COASTAL,DEFENDED,2030,SSP3_7.0,FATHOM/FLOOD_MAP-1ARCSEC-NW_OFFSET-1in100-COAS...
69,1in100,COASTAL,DEFENDED,2030,SSP5_8.5,FATHOM/FLOOD_MAP-1ARCSEC-NW_OFFSET-1in100-COAS...
70,1in100,COASTAL,DEFENDED,2050,SSP1_2.6,FATHOM/FLOOD_MAP-1ARCSEC-NW_OFFSET-1in100-COAS...
71,1in100,COASTAL,DEFENDED,2050,SSP2_4.5,FATHOM/FLOOD_MAP-1ARCSEC-NW_OFFSET-1in100-COAS...
72,1in100,COASTAL,DEFENDED,2050,SSP3_7.0,FATHOM/FLOOD_MAP-1ARCSEC-NW_OFFSET-1in100-COAS...
73,1in100,COASTAL,DEFENDED,2050,SSP5_8.5,FATHOM/FLOOD_MAP-1ARCSEC-NW_OFFSET-1in100-COAS...
74,1in100,COASTAL,DEFENDED,2080,SSP1_2.6,FATHOM/FLOOD_MAP-1ARCSEC-NW_OFFSET-1in100-COAS...


In [5]:
# Get a list of all tiles in one scenario
from shapely import box

# get a list of all fully processed tiles
out_files = os.listdir(out_folder)
all_res = {}
for out_f in out_files:
    tile = out_f.split("_")[0]
    try:
        all_res[tile].append(out_f)
    except:
        all_res[tile] = [out_f]

processed_tiles = []
for key, value in all_res.items():
    if len(value) == 39:
        processed_tiles.append(f"{key}.tif")


path_prefix = "FATHOM/v31/FLOOD_MAP-1ARCSEC-NW_OFFSET-1in100-{hazard}-DEFENDED-DEPTH-{year}-{scenario}-v3.1"

# get a list of all tif files in the s3 bucket for the specified hazard, year, and scenario
paginator = s3_client.get_paginator('list_objects_v2')
tif_files = []
for page in paginator.paginate(Bucket=s3_bucket, Prefix=path_prefix.format(hazard="FLUVIAL", year=2020, scenario="PERCENTILE50")):
    if 'Contents' in page:
        for obj in page['Contents']:
            key = obj['Key']
            # Check for .tif or .tiff extensions
            if key.lower().endswith(('.tif', '.tiff')):
                tif_files.append(key.split('/')[-1])  # Get just the filename

print(f"Total tiles: {len(tif_files)}")
print(f"Processed tiles: {len(processed_tiles)}")
tif_files = list(set(tif_files) - set(processed_tiles))

Total tiles: 19251
Processed tiles: 996


In [ ]:

with rasterio.Env(GDAL_HTTP_UNSAFESSL='YES'):
    for ghs_pop_file in ghs_pop_files:
        pop_file_name = ghs_pop_file.split("_")[-3]
        ghs_r = rasterio.open(ghs_pop_file)        

        scenarios = ['PERCENTILE50'] #vrt_df['scenario'].unique()        
        years = ['2020'] #vrt_df['year'].unique()
        depth_thresh = [0,15,50]
        
        for tile in tif_files:
            # Extract the population for the selected tile
            example_fluvial_path = "s3://{bucket}/{path}/{tile}".format(bucket=s3_bucket, path=path_prefix.format(hazard="FLUVIAL", year=2020, scenario="PERCENTILE50"), tile=tile)
            fluvial_r = rasterio.open(example_fluvial_path)
            fluvial_meta = fluvial_r.meta.copy()
            
            # get boundaing box of the raster
            tile_box = box(*fluvial_r.bounds)
            
            # Turn the tile_box shape into a geodataframe and reproject to the same crs as the ghs raster
            tile_gdf = gpd.GeoDataFrame(geometry=[tile_box], crs=fluvial_r.crs)
            tile_gdf = tile_gdf.to_crs(ghs_r.crs)
            ghs_data, ghs_meta = rMisc.clipRaster(ghs_r, tile_gdf, None, True)
            with rMisc.create_rasterio_inmemory(ghs_meta, ghs_data) as ghs_local:                        
                for scenario in scenarios:
                    if not "PERCENTILE50" in scenario:
                        scenario = f"{scenario}-PERCENTILE50"
                    for year in years:                
                        fluvial_path = "s3://{bucket}/{path}/{tile}".format(bucket=s3_bucket, path=path_prefix.format(hazard="FLUVIAL", year=year, scenario=scenario), tile=tile)
                        coastal_path = "s3://{bucket}/{path}/{tile}".format(bucket=s3_bucket, path=path_prefix.format(hazard="COASTAL", year=year, scenario=scenario), tile=tile)
                        pluvial_path = "s3://{bucket}/{path}/{tile}".format(bucket=s3_bucket, path=path_prefix.format(hazard="PLUVIAL", year=year, scenario=scenario), tile=tile)
                        tPrint(f"Processing tile {tile} for year {year} and scenario {scenario}...")                                                    
                        try:
                            fluvial_r = rasterio.open(fluvial_path)
                            fluvial_meta = fluvial_r.meta.copy()    
                            # Stack the rasters together and take the max value across the stack to get the combined flood depth
                            fluvial_data = fluvial_r.read()
                            pluvial_data = rasterio.open(pluvial_path).read()
                            max_depth = np.maximum.reduce([fluvial_data, pluvial_data])
                            try:
                                coastal_data = rasterio.open(coastal_path).read()
                                max_depth = np.maximum.reduce([max_depth, coastal_data])
                            except:
                                pass                            
                            for cDepth in depth_thresh:
                                out_file = os.path.join(out_folder, f"{tile[:-4]}_FATHOM_{year}_{scenario}_{cDepth}cm_{pop_file_name}m_proportion.tif")
                                if not os.path.exists(out_file):            
                                    numerator = np.where(max_depth > cDepth, 1, 0)
                                    denominator = np.where(max_depth > cDepth, 0, 1)
                                    with rMisc.create_rasterio_inmemory(fluvial_meta, numerator[0,:,:]) as fathom_depth:
                                        numerator_scaled, numerator_meta = rMisc.standardizeInputRasters(fathom_depth, ghs_local, resampling_type="sum")
                                    with rMisc.create_rasterio_inmemory(fluvial_meta, denominator[0,:,:]) as fathom_depth:
                                        denominator_scaled, denominator_meta = rMisc.standardizeInputRasters(fathom_depth, ghs_local, resampling_type="sum")
                                    
                                    results = numerator_scaled / (denominator_scaled + numerator_scaled)
                                    numerator_meta.update({"dtype": rasterio.float32, "count": 1})                    
                                    with rasterio.open(out_file, "w", **numerator_meta) as dest:
                                        dest.write(results.astype(rasterio.float32))                                        
                        except:
                            pass

                    

# Check output

In [ ]:
# Choose a specific scenario and map all the tiles to that.

all_result_tiles = os.listdir(out_folder)
def get_tile_info(tile_name):
    parts = tile_name.split("_")
    scenario = parts[3]
    if not "PERCENTILE50" in scenario:
        scenario = "_".join([parts[3], parts[4]])
    return {
        "tile": parts[0],
        "year": parts[2],
        "scenario": scenario,
        "depth": parts[-3],
        "pop_file": parts[-2],
        "filename":tile_name
    }

split_tile_res = [get_tile_info(x) for x in all_result_tiles]
split_tile_df = pd.DataFrame(split_tile_res)
split_tile_df.head()

In [ ]:
# get tile extents of a model
all_res = []
extents_folder = os.path.join(local_folder, "FATHOM_extents")
if not os.path.exists(extents_folder):
    os.makedirs(extents_folder)
for lbl, grp in split_tile_df.groupby(["year", "scenario", "depth", "pop_file"]):
    year, scenario, depth, pop_file = lbl
    name = "_".join(lbl)
    # get box for raster tile
    grp['geometry'] = grp['filename'].apply(lambda x: box(*rasterio.open(os.path.join(out_folder, x)).bounds))

    crs = rasterio.open(os.path.join(out_folder, grp['filename'].iloc[0])).crs
    grp_gdf = gpd.GeoDataFrame(grp, geometry='geometry', crs=crs)
    grp_gdf.to_file(os.path.join(extents_folder, f"{name}.gpkg"), driver="GPKG")
    model_shape = grp_gdf.unary_union

    all_res.append([name, model_shape])
        
final_shape = gpd.GeoDataFrame(all_res, columns=["name", "geometry"], crs=crs)
final_shape.to_file(os.path.join(local_folder, "FATHOM_model_extents.gpkg"), driver="GPKG")

In [ ]:
### The current process prodcued a ton of files with trash georeferencing, so we need to check the output of the combine and scale process.
# Get a list of all output files for the FATHOM combine and scale process

year = '2020'
scenario = "PERCENTILE50"
depth = "50cm"
pop_file = "1000m"

sel_model = split_tile_df[(split_tile_df['year'] == year) & (split_tile_df['scenario'] == scenario) & (split_tile_df['depth'] == depth) & (split_tile_df['pop_file'] == pop_file)]
sel_model.sort_values(by="tile", inplace=True)
sel_model

In [ ]:
aws_path = "s3://wbg-geography01/FATHOM/v31/FLOOD_MAP-1ARCSEC-NW_OFFSET-1in{return_p}-PLUVIAL-DEFENDED-DEPTH-{year}-{scenario}-v3.1/{tile}.tif"

for idx, row in sel_model.iterrows():
    if idx > 25000:
        break
    sel_path = aws_path.format(return_p=100, year=year, scenario=scenario, tile=row['tile'])
    local_path = os.path.join(out_folder, row['filename'])
    

In [ ]:
sel_model['tile'].values

In [ ]:
row = sel_model.loc[sel_model['tile'] == 'n27w082'].iloc[0]
sel_path = aws_path.format(return_p=100, year=year, scenario=scenario, tile=row['tile'])
local_path = os.path.join(out_folder, row['filename'])

In [ ]:
with rasterio.Env(GDAL_HTTP_UNSAFESSL='YES'):
    xx = rasterio.open(sel_path)
    print(xx.bounds)


In [ ]:
yy = rasterio.open(local_path)
yy = rMisc.project_raster(yy, xx.crs)

In [ ]:
local_path

In [ ]:
row

In [ ]:
rMisc.project_raster?